## **Logic Gate Minimization**
Implement Quine-McCluckey Algorithm via detecting prime implicants and essential prime implicants

Input: Sum-of-Products form

Output: Minimised logic expression, associated K-map

Quine-McCluckey algorithm: [Link](https://en.wikipedia.org/wiki/Quine%E2%80%93McCluskey_algorithm)

We can get the most optimal logic gate expression from all possible additions involving the essential implicants and the remaining prime implicants by using Petrick's method, but it is too complicated for now.

In [12]:
from itertools import product
from typing import List, Dict, Set, Tuple

In [13]:
# generalized grey code
# add zero in front of previous grey code dist, flip it and add a 1 in front it to get the other half

def generate_gray_code(n: int) -> List[str]:
    """
    Generate n-bit Gray code sequence.
    """
    if n == 0:
        return [""]
    if n == 1:
        return ["0", "1"]

    previous = generate_gray_code(n - 1)

    first_half = ["0" + code for code in previous]
    second_half = ["1" + code for code in reversed(previous)]

    return first_half + second_half

In [14]:
# Input the minterm expression
# takes number of variables from the variable list and takes minterm positions as well
#max minterm can only be 2^n-1, and hence the limit
# also takes don't care terms (positions where output value doesn't matter)

def get_minterm_input() -> tuple[int, List[str], List[int], List[int]]:
    """
    Returns tuple (number_of_variables, variable_names, sorted_minterms, sorted_dont_cares)
    """

    variable_names = input("Enter variable names (space-separated): ").split()

    if not variable_names:
        raise ValueError("At least one variable must be provided.")

    n = len(variable_names)

    minterms = list(map(int, input("Enter minterms (space-separated): ").split()))

    dont_cares_raw = input("Enter don't care terms (space-separated, leave blank if none): ").split()
    dont_cares = list(map(int, dont_cares_raw)) if dont_cares_raw else []

    limit = 2 ** n

    if any(m < 0 or m >= limit for m in minterms):
        raise ValueError(f"Minterms must be between 0 and {limit - 1}.")

    if any(d < 0 or d >= limit for d in dont_cares):
        raise ValueError(f"Don't care terms must be between 0 and {limit - 1}.")

    minterms = sorted(set(minterms))
    dont_cares = sorted(set(dont_cares) - set(minterms))

    return n, variable_names, minterms, dont_cares

In [15]:
# truth table generation
# takes in the minterms with the value one, and don't cares (marked "X")
# gives a truth table back

def generate_truth_table(n: int, minterms: List[int], dont_cares: List[int] = None) -> List[List]:

    dont_cares = dont_cares or []

    truth_table = []

    for bits in product([0, 1], repeat=n):

        decimal = 0
        for bit in bits:
            decimal = decimal * 2 + bit

        if decimal in minterms:
            output = 1
        elif decimal in dont_cares:
            output = "X"
        else:
            output = 0

        truth_table.append(list(bits) + [output])

    return truth_table

#show the truth table
def print_truth_table(variable_names: List[str], truth_table: List[List[int]]) -> None:

    header = " ".join(variable_names) + " | F"

    print(header)
    print("-" * len(header))

    for row in truth_table:
        print(" ".join(map(str, row[:-1])), "|", row[-1])

In [16]:
# k-map generation
def generate_kmap(n: int, minterms: List[int], dont_cares: List[int] = None):

    dont_cares = dont_cares or []

    row_bits = n // 2
    col_bits = n - row_bits

    row_gray = generate_gray_code(row_bits)
    col_gray = generate_gray_code(col_bits)

    kmap = []

    for r in row_gray:

        row = []

        for c in col_gray:

            binary = r + c
            decimal = int(binary, 2)

            if decimal in minterms:
                row.append(1)
            elif decimal in dont_cares:
                row.append("X")
            else:
                row.append(0)

        kmap.append(row)

    return row_gray, col_gray, kmap

def print_kmap(variable_names, row_gray, col_gray, kmap):

    row_bits = len(row_gray[0])
    row_label = "".join(variable_names[:row_bits])
    col_label = "".join(variable_names[row_bits:])

    print(" " * (len(row_label) + 5) + col_label)
    print(" " * (len(row_label) + 5) + " ".join(col_gray))
    print(" " * (len(row_label) + 3) + "-" * (len(" ".join(col_gray)) + 2))

    for i, row in enumerate(kmap):
        if i == 0:
            print(f"{row_label} {row_gray[i]:>2} |", end=" ")
        else:
            print(f"{' ' * len(row_label)} {row_gray[i]:>2} |", end=" ")

        print("  ".join(map(str, row)))

In [17]:
def decimal_to_binary(minterms: List[int], n: int) -> List[str]:
    return [format(m, f"0{n}b") for m in minterms]

In [18]:
def group_minterms(binary_terms: List[str]) -> Dict[int, List[str]]:

    groups = {}

    for term in binary_terms:

        ones = term.count("1")

        if ones not in groups:
            groups[ones] = []

        groups[ones].append(term)

    return groups

In [20]:
#signed 2 combination procedure

def combine_terms(term1: str, term2: str) -> tuple[bool, str]:

    differences = 0
    combined = []

    for a, b in zip(term1, term2):

        if a == b:
            combined.append(a)
        else:
            differences += 1
            combined.append("-")

        if differences > 1:
            return False, ""

    return differences == 1, "".join(combined)

In [21]:
#combine adjacent groups in preparation of signed 4 combination

def combine_groups(groups: Dict[int, List[str]]) -> tuple[Dict[int, List[str]], Set[str], Set[str]]:

    new_groups = {}
    used_terms = set()
    prime_implicants = set()

    group_keys = sorted(groups.keys())

    for i in range(len(group_keys) - 1):

        current = groups[group_keys[i]]
        next_group = groups[group_keys[i + 1]]

        for term1 in current:
            for term2 in next_group:

                success, combined = combine_terms(term1, term2)

                if success:

                    used_terms.add(term1)
                    used_terms.add(term2)

                    ones = combined.count("1")

                    if ones not in new_groups:
                        new_groups[ones] = []

                    if combined not in new_groups[ones]:
                        new_groups[ones].append(combined)

    # Anything not used is a prime implicant
    for group in groups.values():
        for term in group:
            if term not in used_terms:
                prime_implicants.add(term)

    return new_groups, used_terms, prime_implicants

In [22]:
# driver function to call combine_groups until all prime implicants are found
# don't cares are included when forming/combining groups (they can help produce
# larger, simpler groups) but -- since they don't need to be covered -- they are
# left out of the prime implicant chart later (build_prime_chart only uses minterms)

def find_prime_implicants(minterms: List[int], n: int, dont_cares: List[int] = None) -> List[str]:

    dont_cares = dont_cares or []

    # Initial grouping (minterms + don't cares combined)
    all_terms = sorted(set(minterms) | set(dont_cares))
    binary_terms = decimal_to_binary(all_terms, n)
    groups = group_minterms(binary_terms)

    prime_implicants = set()

    while True:

        new_groups, _, new_primes = combine_groups(groups)

        # Store newly found prime implicants
        prime_implicants.update(new_primes)

        # No more combinations possible
        if not new_groups:
            break

        groups = new_groups

    return sorted(prime_implicants)

In [23]:
def implicant_covers(implicant: str, minterm: str) -> bool:
    for imp_bit, min_bit in zip(implicant, minterm):
        if imp_bit == "-":
            continue
        if imp_bit != min_bit:
            return False
    return True

In [24]:
# builds the chart
def build_prime_chart(prime_implicants: List[str],
                      minterms: List[int],
                      n: int) -> Dict[int, List[str]]:

    chart = {}

    binary_minterms = decimal_to_binary(minterms, n)

    for decimal, binary in zip(minterms, binary_minterms):

        chart[decimal] = []

        for implicant in prime_implicants:

            if implicant_covers(implicant, binary):
                chart[decimal].append(implicant)

    return chart

def print_prime_chart(chart):

    print("Prime Implicant Chart\n")

    for minterm in chart:
        print(f"{minterm:>3} : {chart[minterm]}")

In [25]:
def find_essential_prime_implicants(chart: Dict[int, List[str]]) -> Set[str]:
    essential = set()

    for implicants in chart.values():

        if len(implicants) == 1:
            essential.add(implicants[0])

    return essential

In [27]:
def remove_covered_minterms(chart: Dict[int, List[str]],
                            essential: Set[str]) -> Dict[int, List[str]]:
    reduced_chart = {}

    for minterm, implicants in chart.items():

        covered = False

        for term in essential:

            if term in implicants:
                covered = True
                break

        if not covered:
            reduced_chart[minterm] = implicants

    return reduced_chart

In [28]:
from itertools import product, combinations

In [29]:
# some more helper functions
def find_all_covers(reduced_chart: Dict[int, List[str]]) -> List[List[str]]:
    if not reduced_chart:
        return [[]]

    # Unique remaining implicants
    implicants = sorted({
        implicant
        for terms in reduced_chart.values()
        for implicant in terms
    })

    minterms = list(reduced_chart.keys())

    valid_covers = []

    # Try every subset of implicants
    for r in range(1, len(implicants) + 1):

        for subset in combinations(implicants, r):

            covers_all = True

            for minterm in minterms:

                # At least one implicant in the subset must cover this minterm
                if not any(implicant in reduced_chart[minterm]
                           for implicant in subset):

                    covers_all = False
                    break

            if covers_all:
                valid_covers.append(list(subset))

    return valid_covers

In [30]:
def minimum_covers(covers: List[List[str]]) -> List[List[str]]:

    if not covers:
        return []

    minimum = min(len(c) for c in covers)

    return [c for c in covers if len(c) == minimum]

In [31]:
def print_cover_options(essential: Set[str],
                        covers: List[List[str]]) -> None:

    print("\nEssential Prime Implicants")

    for implicant in sorted(essential):
        print(implicant)

    print("\nPossible Minimum Covers")
    print("-----------------------")

    if not covers:
        print("No additional implicants required.")
        return

    for i, cover in enumerate(covers, start=1):

        print(f"\nOption {i}")

        for implicant in sorted(essential):
            print(implicant)

        for implicant in cover:
            print(implicant)

In [32]:
def implicant_to_expression(implicant: str,
                            variables: List[str]) -> str:


    expression = ""

    for bit, variable in zip(implicant, variables):

        if bit == "1":
            expression += variable

        elif bit == "0":
            expression += variable + "'"

        # Ignore don't-care bits ('-')

    return expression if expression else "1"

In [33]:
def solution_to_expression(essential: Set[str],
                           additional: List[str],
                           variables: List[str]) -> str:
    """
    Convert a complete cover into a Boolean SOP expression.
    """

    terms = []

    for implicant in sorted(essential):
        terms.append(implicant_to_expression(implicant, variables))

    for implicant in additional:
        terms.append(implicant_to_expression(implicant, variables))

    return " + ".join(terms)

In [34]:
def print_cover_options(essential: Set[str],
                        covers: List[List[str]],
                        variables: List[str]) -> None:

    print("\nEssential Prime Implicants")
    print("--------------------------")

    for implicant in sorted(essential):
        print(f"{implicant:6} -> {implicant_to_expression(implicant, variables)}")

    print("\nPossible Simplified Expressions")
    print("-------------------------------")

    if not covers:
        print("F =", solution_to_expression(essential, [], variables))
        return

    for i, cover in enumerate(covers, start=1):

        expression = solution_to_expression(
            essential,
            cover,
            variables
        )

        print(f"\nOption {i}")
        print(f"F = {expression}")

In [35]:
# --------------------------
# Input
# --------------------------

n, variables, minterms, dont_cares = get_minterm_input()

# --------------------------
# Truth Table
# --------------------------

print("\n" + "="*60)
print("TRUTH TABLE")
print("="*60)

truth_table = generate_truth_table(n, minterms, dont_cares)
print_truth_table(variables, truth_table)

# --------------------------
# Karnaugh Map
# --------------------------

print("\n" + "="*60)
print("KARNAUGH MAP")
print("="*60)

row_gray, col_gray, kmap = generate_kmap(n, minterms, dont_cares)
print_kmap(variables, row_gray, col_gray, kmap)

# --------------------------
# Prime Implicants
# --------------------------

print("\n" + "="*60)
print("PRIME IMPLICANTS")
print("="*60)

prime_implicants = find_prime_implicants(minterms, n, dont_cares)

for implicant in prime_implicants:
    print(implicant)

# --------------------------
# Prime Implicant Chart
# --------------------------

print("\n" + "="*60)
print("PRIME IMPLICANT CHART")
print("="*60)

chart = build_prime_chart(prime_implicants, minterms, n)
print_prime_chart(chart)

# --------------------------
# Essential Prime Implicants
# --------------------------

print("\n" + "="*60)
print("ESSENTIAL PRIME IMPLICANTS")
print("="*60)

essential = find_essential_prime_implicants(chart)

for implicant in sorted(essential):
    print(f"{implicant:6} -> {implicant_to_expression(implicant, variables)}")

# --------------------------
# Remaining Minterms
# --------------------------

print("\n" + "="*60)
print("UNCOVERED MINTERMS")
print("="*60)

reduced_chart = remove_covered_minterms(chart, essential)

if not reduced_chart:
    print("All minterms are covered.")
else:
    print_prime_chart(reduced_chart)

# --------------------------
# All Possible Covers
# --------------------------

print("\n" + "="*60)
print("SIMPLIFIED BOOLEAN EXPRESSIONS")
print("="*60)

cover_options = find_all_covers(reduced_chart)
cover_options = minimum_covers(cover_options)

print_cover_options(essential, cover_options, variables)


TRUTH TABLE
A B C | F
---------
0 0 0 | 1
0 0 1 | 0
0 1 0 | 1
0 1 1 | 1
1 0 0 | 0
1 0 1 | 0
1 1 0 | 1
1 1 1 | 1

KARNAUGH MAP
      BC
      00 01 11 10
    -------------
A  0 | 1  0  1  1
   1 | 0  0  1  1

PRIME IMPLICANTS
-1-
0-0

PRIME IMPLICANT CHART
Prime Implicant Chart

  0 : ['0-0']
  2 : ['-1-', '0-0']
  3 : ['-1-']
  6 : ['-1-']
  7 : ['-1-']

ESSENTIAL PRIME IMPLICANTS
-1-    -> B
0-0    -> A'C'

UNCOVERED MINTERMS
All minterms are covered.

SIMPLIFIED BOOLEAN EXPRESSIONS

Essential Prime Implicants
--------------------------
-1-    -> B
0-0    -> A'C'

Possible Simplified Expressions
-------------------------------

Option 1
F = B + A'C'
